# PatchTST trên PTB-XL: train + Membership Inference (TV2, Hoàng), bản Kaggle v2

Pipeline: PTB-XL → split member / non-member chung của nhóm → tune trên `val` → train **target** → train **8 shadow** → tấn công **Loss / Conf / Entropy, LiRA, RMIA, Temporal** → TPR @ 0,1% và 1% FPR + bootstrap CI → ghi `results/patchtst/<task>/<attack>.json` vào `/kaggle/working`.

| Bước | Mục tiêu | Vai trò trong MIA |
|---|---|---|
| 1–3 | Đường dẫn, cấu hình, đọc dữ liệu | Dùng đúng split chung của nhóm (kiểm sha256) |
| 4–5 | Model PatchTST + recipe train | Giống nhau cho target và shadow |
| 6 | Tune ≤ 6 cấu hình trên `val` | Chọn (epochs, max_lr) mà **không nhìn điểm MIA** |
| 7 | Train **target** trên member | Đây là "nạn nhân" bị tấn công |
| 8 | Train **8 shadow** | Mô hình tham chiếu cho LiRA / RMIA |
| 9–12 | Các attack | Chấm điểm "có phải member không" |
| 13–15 | Bảng, ROC, hình demo, zip | Đưa kết quả về repo |

**Chuẩn bị (một lần)**
1. **Add Input → Datasets**, tìm `khyeh0719/ptb-xl-dataset` (PTB-XL v1.0.1).
2. Không cần clone repo: notebook chứa sẵn `make_split.py`, tự tạo lại split rồi so sha256 với bản của repo. Lệch hash thì dừng.
3. **Settings → Accelerator → GPU T4 x2** (notebook dùng 1 GPU). Internet bật hay tắt đều được (`transformers` có sẵn trên Kaggle; thiếu `wfdb` thì tự đọc file `.dat`).

**Chạy**
- Học / thử: chạy từng cell (Shift+Enter) trong phiên tương tác. Muốn thử nhanh: ở cell cấu hình đặt `RUN_TUNE = False` (dùng luôn cấu hình đã tune ngày 18/10).
- Chạy thật: **Save Version → Save & Run All (Commit)**. Thời gian trên T4 (theo log 18/10): **cls ≈ 1 giờ 45** (tune 53′, target 6′, 8 shadow 41′), **fcst ≈ 2 giờ 40** (tune 49′, target 12′, 8 shadow 94′).
- Bị dừng giữa chừng: **Add Input → Notebook Output**, chọn version trước, rồi chạy lại. Cell "Chạy tiếp" chép checkpoint cũ vào và bỏ qua phần đã xong (từng cấu hình tune, target, từng shadow). Output của notebook v1 (`work/patchtst_<task>/`) cũng dùng được.

Quy ước xuyên suốt: **score càng cao = càng giống member.**

In [ ]:
try:                                       # PTB-XL is read with wfdb if available, otherwise with the built-in reader (Cell 3)
    import wfdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=False)   # fails quietly when Internet is off

## 1. Đường dẫn + split chung của nhóm
PTB-XL được tìm trong `/kaggle/input`. Split chung (member = `train`, non-member = `test`, `val` = 500 bản ghi để tune) được tạo lại từ PTB-XL bằng chính `make_split.py` của repo (nhúng trong cell, giống hệt notebook TV1) vào `/tmp/patchtst_dataset`, và phải khớp sha256 với bản trong repo. Như vậy cả nhóm chắc chắn tấn công trên **cùng một tập member / non-member**.

Checkpoint và kết quả ghi vào `/kaggle/working` (= output của version). Cache tín hiệu (~0,5 GB) để ở `/tmp` cho khỏi lọt vào output.

In [ ]:
import os, sys, glob, json, time, math, random, hashlib, subprocess, shutil
import numpy as np
KAGGLE_INPUT = os.environ.get('PTST_KAGGLE_INPUT', '/kaggle/input')
WORK_ROOT = os.environ.get('PTST_WORK', '/kaggle/working')          # ckpt/ and results/ become the notebook output
LOCAL_CACHE = os.environ.get('PTST_CACHE', '/tmp/cache_patchtst')
DATASET_DIR = os.environ.get('PTST_DATASET_DIR', '/tmp/patchtst_dataset')  # make_split.py + splits/, rebuilt here
os.makedirs(f'{DATASET_DIR}/splits', exist_ok=True)

def find_input(name):
    """First file called `name` under /kaggle/input (skips the 40k-file PTB-XL record folders)."""
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        dirs[:] = sorted(d for d in dirs if d not in ('records100', 'records500'))
        if name in files:
            return os.path.join(root, name)

csv = find_input('ptbxl_database.csv')
assert csv, f'PTB-XL not found under {KAGGLE_INPUT}: Add Input -> Datasets -> khyeh0719/ptb-xl-dataset'
PTBXL_ROOT = os.path.dirname(csv)                                   # PTB-XL is already unpacked by Kaggle
print('PTB-XL:', PTBXL_ROOT)

# ---- the group split, rebuilt from PTB-XL with the repo's own code (no extra upload needed) ----
# verbatim copy of dataset/make_split.py
MAKE_SPLIT_SRC = '''"""Shared PTB-XL member / non-member split for all models (LSTM, TCN, PatchTST, iTransformer, MOMENT).

Uses only the 5-superclass, single-label records (same filter as the MOMENT tutorial), so every
team member gets identical ecg_id sets. Output keys follow the MOMENT dataset convention:

    train = member       (used to train the target model)
    test  = non-member   (never seen by the target model)
    val   = small held-out set carved out of the non-member pool (not used for the attack)

Usage:
    python make_split.py --ptbxl-dir /path/to/ptb-xl --level both --seed 42
"""
import argparse
import ast
import hashlib
import json
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

CLASSES = ['CD', 'HYP', 'MI', 'NORM', 'STTC']   # sorted, same as sklearn LabelEncoder in momentfm


def load_single_label_records(ptbxl_dir):
    df = pd.read_csv(os.path.join(ptbxl_dir, 'ptbxl_database.csv'), index_col='ecg_id')
    codes = pd.read_csv(os.path.join(ptbxl_dir, 'scp_statements.csv'), index_col=0)
    diag = codes[codes.diagnostic == 1.0].diagnostic_class
    df['scp_codes'] = df.scp_codes.apply(ast.literal_eval)
    df['superclass'] = df.scp_codes.apply(lambda d: sorted({diag[k] for k in d if k in diag.index and pd.notna(diag[k])}))
    df = df[df.superclass.apply(len) == 1].copy()
    df['superclass'] = df.superclass.apply(lambda c: c[0])
    assert set(df.superclass) <= set(CLASSES), set(df.superclass)
    df['label'] = df.superclass.map({c: i for i, c in enumerate(CLASSES)})
    return df.sort_index()


def make_split(df, level, seed, n_val):
    ids = df.index.values
    if level == 'record':
        member, non = train_test_split(ids, test_size=0.5, stratify=df.label.values, random_state=seed)
    elif level == 'user':
        gss = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
        mi, ni = next(gss.split(ids, groups=df.patient_id.values))
        member, non = ids[mi], ids[ni]
        pid = df.patient_id
        assert not (set(pid.loc[member]) & set(pid.loc[non])), 'patient leak between member and non-member'
    else:
        raise ValueError(level)
    non = np.random.default_rng(seed).permutation(non)
    split = {'train': sorted(int(i) for i in member),
             'val': sorted(int(i) for i in non[:n_val]),
             'test': sorted(int(i) for i in non[n_val:])}
    assert not (set(split['train']) & set(split['val'])) and not (set(split['train']) & set(split['test']))
    assert not (set(split['val']) & set(split['test']))
    return split


def make_shadow_masks(df, level, seed, k):
    """Balanced in/out masks for LiRA / RMIA shadow models over the whole single-label pool.

    masks[j, i] = True  <=>  shadow j is trained on record ids[i]. Every unit (record, or patient at
    user level) is "in" for exactly k/2 shadows, so each sample has k/2 IN and k/2 OUT reference logits.
    Shadows use the same training recipe (epochs, optimizer, lr) as the target.
    """
    assert k % 2 == 0, 'k must be even'
    ids = df.index.values
    unit = ids if level == 'record' else df.patient_id.values
    units = np.unique(unit)
    rng = np.random.default_rng(seed + 1000)
    unit_masks = np.zeros((k, len(units)), dtype=bool)
    for u in range(len(units)):
        unit_masks[rng.permutation(k)[:k // 2], u] = True
    masks = unit_masks[:, np.searchsorted(units, unit)]
    ref_ids = np.sort(rng.choice(ids, size=min(5000, len(ids)), replace=False))   # RMIA population Z
    return ids, masks, ref_ids


def write_shadow(df, level, seed, k, out_dir):
    ids, masks, ref_ids = make_shadow_masks(df, level, seed, k)
    stem = f'ptbxl_{level}_s{seed}'
    path = os.path.join(out_dir, f'shadow_masks_{level}_K{k}_s{seed}.npz')
    np.savez_compressed(path, ids=ids, masks=masks, ref_ids=ref_ids)
    with open(path, 'rb') as f:
        digest = hashlib.sha256(f.read()).hexdigest()
    meta_path = os.path.join(out_dir, stem + '.meta.json')
    with open(meta_path) as f:
        meta = json.load(f)
    meta['shadow'] = {'file': os.path.basename(path), 'K': k, 'sha256': digest, 'n_ref': int(len(ref_ids)),
                      'in_fraction': float(masks.mean()),
                      'rmia_defaults': {'gamma': 2.0, 'a': 0.3, 'offline': True}}
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=1)
    print(' ', os.path.basename(path), masks.shape, 'sha256', digest[:12])


def write(df, split, level, seed, n_val, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    stem = f'ptbxl_{level}_s{seed}'
    path = os.path.join(out_dir, stem + '.json')
    with open(path, 'w') as f:
        json.dump(split, f)
    with open(path, 'rb') as f:
        digest = hashlib.sha256(f.read()).hexdigest()
    meta = {
        'level': level, 'seed': seed, 'n_val': n_val, 'classes': CLASSES, 'sha256': digest,
        'n_records_total': int(len(df)), 'n_patients_total': int(df.patient_id.nunique()),
        'sizes': {k: len(v) for k, v in split.items()},
        'class_counts': {k: np.bincount(df.label.loc[v].values, minlength=len(CLASSES)).tolist() for k, v in split.items()},
        'n_patients': {k: int(df.patient_id.loc[v].nunique()) for k, v in split.items()},
    }
    with open(os.path.join(out_dir, stem + '.meta.json'), 'w') as f:
        json.dump(meta, f, indent=1)
    print(stem, meta['sizes'], 'sha256', digest[:12])
    if meta['sizes']['test'] < 10000:
        print(f"  WARNING: only {meta['sizes']['test']} non-members; TPR@0.1%FPR rests on ~{int(meta['sizes']['test'] * 0.001)} samples")


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--ptbxl-dir', required=True, help='folder containing ptbxl_database.csv')
    ap.add_argument('--level', choices=['record', 'user', 'both'], default='both')
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--n-val', type=int, default=500)
    ap.add_argument('--out-dir', default=os.path.join(os.path.dirname(os.path.abspath(__file__)), 'splits'))
    ap.add_argument('--shadow-k', type=int, default=8, help='number of LiRA/RMIA shadow masks (even); 0 to skip')
    a = ap.parse_args()
    data = load_single_label_records(a.ptbxl_dir)
    for lv in (['record', 'user'] if a.level == 'both' else [a.level]):
        write(data, make_split(data, lv, a.seed, a.n_val), lv, a.seed, a.n_val, a.out_dir)
        if a.shadow_k:
            write_shadow(data, lv, a.seed, a.shadow_k, a.out_dir)
'''
# verbatim copy of dataset/splits/ptbxl_record_s42.meta.json (sha256 of the official split + shadow masks)
SPLIT_META = json.loads('''{
 "level": "record",
 "seed": 42,
 "n_val": 500,
 "classes": [
  "CD",
  "HYP",
  "MI",
  "NORM",
  "STTC"
 ],
 "sha256": "35dd8ca57797bed344db4bc8ff0f99b6f05e78add6c565bed54932d5687adce7",
 "n_records_total": 16272,
 "n_patients_total": 14641,
 "sizes": {
  "train": 8136,
  "val": 500,
  "test": 7636
 },
 "class_counts": {
  "train": [
   854,
   268,
   1269,
   4542,
   1203
  ],
  "val": [
   54,
   24,
   91,
   266,
   65
  ],
  "test": [
   801,
   244,
   1178,
   4275,
   1138
  ]
 },
 "n_patients": {
  "train": 7696,
  "val": 497,
  "test": 7211
 },
 "shadow": {
  "file": "shadow_masks_record_K8_s42.npz",
  "K": 8,
  "sha256": "fbda57864ba16e9d50fd1bcb1c1c5287548511dc476a02924d494c36aa2a5dc9",
  "n_ref": 5000,
  "in_fraction": 0.5,
  "rmia_defaults": {
   "gamma": 2.0,
   "a": 0.3,
   "offline": true
  }
 }
}''')
MASK_CONTENT_SHA256 = 'e3b5d6ccea6ee815f60e577b51a978b9928db6bcfe790e8251a01eee01795faa'   # sha256 of ids | masks | ref_ids

open(f'{DATASET_DIR}/make_split.py', 'w').write(MAKE_SPLIT_SRC)
sys.path.insert(0, DATASET_DIR)
import make_split

split_file = f'{DATASET_DIR}/splits/ptbxl_record_s42.json'
mask_file = f'{DATASET_DIR}/splits/shadow_masks_record_K8_s42.npz'
df_split = make_split.load_single_label_records(PTBXL_ROOT)
with open(split_file, 'w') as f:                                    # same bytes as make_split.write()
    json.dump(make_split.make_split(df_split, 'record', SPLIT_META['seed'], SPLIT_META['n_val']), f)
sha = hashlib.sha256(open(split_file, 'rb').read()).hexdigest()
assert sha == SPLIT_META['sha256'], f'rebuilt split sha256 {sha[:12]} != repo {SPLIT_META["sha256"][:12]}: different PTB-XL copy or sklearn version, stop'

ids, masks, ref_ids = make_split.make_shadow_masks(df_split, 'record', SPLIT_META['seed'], SPLIT_META['shadow']['K'])
h = hashlib.sha256(ids.astype('<i8').tobytes() + masks.astype(bool).tobytes(order='F') + ref_ids.astype('<i8').tobytes()).hexdigest()
assert h == MASK_CONTENT_SHA256, f'rebuilt shadow masks {h[:12]} differ from the repo ones, stop'
np.savez_compressed(mask_file, ids=ids, masks=masks, ref_ids=ref_ids)   # file bytes differ (zip timestamps), content is identical
json.dump(SPLIT_META, open(f'{DATASET_DIR}/splits/ptbxl_record_s42.meta.json', 'w'), indent=1)
del df_split, ids, masks, ref_ids
print('split rebuilt and matches the repo: sha256', sha[:12], SPLIT_META['sizes'], '->', DATASET_DIR, sorted(os.listdir(f'{DATASET_DIR}/splits')))

## 2. Cấu hình
Chỉ cần đổi `TASK`. `EPOCHS` / `MAX_LR` bị ghi đè bằng cấu hình tune tốt nhất ở bước 6; đặt `RUN_TUNE = False` để bỏ qua tune và dùng giá trị bên dưới (đúng bằng kết quả tune 18/10, tiết kiệm ~50 phút).
Phần còn lại (batch 16, AdamW, OneCycleLR, grad clip 1.0, K = 8 shadow, L = 512, H = 96) là bắt buộc theo `dataset/README.md`.

- **cls**: input 512 điểm × 12 lead → dự đoán 1 trong 5 nhóm bệnh (`CD, HYP, MI, NORM, STTC`). Loss = cross-entropy.
- **fcst**: input 512 điểm đầu → dự báo 96 điểm tiếp theo của cả 12 lead. Loss = MSE.

In [ ]:
TASK = os.environ.get('PTST_TASK', 'cls')                 # 'cls' or 'fcst'
RUN_TUNE = os.environ.get('PTST_RUN_TUNE', '1') == '1'    # False: skip Cell 6 and use EPOCHS / MAX_LR below
TUNE_GRID = [(20, 1e-4), (20, 5e-4), (20, 1e-3), (50, 1e-4), (50, 5e-4), (50, 1e-3)]   # <= 6 configs (same as v1)
if os.environ.get('PTST_TUNE_GRID'):                      # e.g. '1:1e-3,2:5e-4' for a quick test
    TUNE_GRID = [(int(e), float(l)) for e, l in (c.split(':') for c in os.environ['PTST_TUNE_GRID'].split(','))]
EPOCHS, MAX_LR = {'cls': (20, 5e-4), 'fcst': (50, 1e-3)}[TASK]   # = configs chosen by the 18/10 tuning run
if os.environ.get('PTST_EPOCHS'):
    EPOCHS, MAX_LR = int(os.environ['PTST_EPOCHS']), float(os.environ['PTST_MAX_LR'])

SEED, LEVEL, BATCH, GRAD_CLIP, N_BOOT = 42, 'record', 16, 1.0, 1000
N_LEADS, RAW_LEN, LOOKBACK, HORIZON = 12, 1000, 512, 96
SEQ_LEN = LOOKBACK + HORIZON          # 608 for both tasks (README section 2); cls only feeds the first 512 points
K = 8                                 # masks are balanced over all 8 shadows only
EVAL_FRACS = (0.1, 0.2, 0.5, 1.0)     # target outputs saved at 10/20/50/100% of EPOCHS (for Temporal)
N_FOLDS, N_BANDS = 5, 8               # Temporal: cross-fitting folds, DFT bands (fcst)
CLASSES = ['CD', 'HYP', 'MI', 'NORM', 'STTC']

def eval_epochs(epochs):
    return sorted({max(1, math.ceil(f * epochs)) for f in EVAL_FRACS})

SPLIT_PATH = f'{DATASET_DIR}/splits/ptbxl_{LEVEL}_s{SEED}.json'
MASK_PATH = f'{DATASET_DIR}/splits/shadow_masks_{LEVEL}_K8_s{SEED}.npz'
CKPT_DIR = f'{WORK_ROOT}/ckpt/patchtst_{TASK}'            # tuning.json, target_ep*.pt, shadow_k*.pt
RESULTS_DIR = f'{WORK_ROOT}/results/patchtst/{TASK}'      # <attack>.json, ROC, demo figure
for d in (CKPT_DIR, RESULTS_DIR, LOCAL_CACHE):
    os.makedirs(d, exist_ok=True)
print('task', TASK, '| tune', RUN_TUNE, TUNE_GRID if RUN_TUNE else (EPOCHS, MAX_LR))

### Chạy tiếp từ version trước (nếu có)
Nếu đã Add Input output của một version trước (v2: `ckpt/patchtst_<task>/`, hoặc v1: `work/patchtst_<task>/`), chép các file đã xong vào `CKPT_DIR`. File đã có thì giữ nguyên. Mỗi file mang một `key` (recipe + sha256 của split); bước sau chỉ dùng lại file khi key khớp, nên chép nhầm cũng không sao.

In [ ]:
def find_dirs(name):
    """Folders called `name` under /kaggle/input (skips the 40k-file PTB-XL record folders)."""
    hits = []
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        dirs[:] = sorted(d for d in dirs if d not in ('records100', 'records500'))
        if os.path.basename(root) == name:
            hits.append(root)
    return hits

KEEP = lambda f: f in ('tuning.json', 'target.done.json') or (f.endswith('.pt') and not f.endswith('.ckpt.pt'))
for prev in find_dirs(f'patchtst_{TASK}'):
    if os.path.abspath(prev) == os.path.abspath(CKPT_DIR):
        continue
    n = 0
    for f in sorted(os.listdir(prev)):
        if KEEP(f) and not os.path.exists(f'{CKPT_DIR}/{f}'):
            shutil.copy(os.path.join(prev, f), CKPT_DIR); n += 1
    print('resumed', n, 'files from', prev)
print('ckpt now:', sorted(os.listdir(CKPT_DIR)))

## 3. Dữ liệu PTB-XL
Kiểm sha256 của split, đọc 16.272 bản ghi single-label (100 Hz → 1000 điểm, 12 lead), nội suy tuyến tính về **608** điểm và z-score theo từng lead của từng bản ghi. cls dùng 512 điểm đầu; fcst dùng 512 điểm đầu làm input và 96 điểm cuối làm đáp án. Lần đầu mất vài phút, sau đó đọc từ cache.

Ba tập chỉ số (đều là vị trí trong `pool_ids`, sắp theo `ecg_id`):
- `idx_m` = **member** (8.136): target được train trên tập này.
- `idx_n` = **non-member** (7.636): target chưa từng thấy.
- `idx_v` = `val` (500): chỉ để tune, không dùng khi tấn công.

`masks_pool[j, i] = True` nghĩa là shadow `j` được train trên bản ghi `i`; mỗi bản ghi là IN của đúng 4 shadow và OUT của 4 shadow còn lại. Nếu một `assert` báo lỗi thì **dừng**: sai bản dataset hoặc sai split.

In [ ]:
import re
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from scipy import interpolate
from concurrent.futures import ThreadPoolExecutor
try:
    import wfdb
except ImportError:
    wfdb = None
    print('wfdb not installed (Internet off?): using the built-in WFDB format-16 reader')

def _read_wfdb_numpy(path):
    """Minimal reader for PTB-XL WFDB records (format 16), used when the wfdb package is missing."""
    with open(path + '.hea') as f:
        lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]
    n_sig, n_samp = int(lines[0].split()[1]), int(lines[0].split()[3])
    gains, bases, dat = [], [], None
    for l in lines[1:1 + n_sig]:
        p = l.split()
        dat = p[0]
        assert p[1] == '16', f'unsupported WFDB format {p[1]} in {path}'
        m = re.match(r'([-\d.eE+]+)(?:\(([-\d]+)\))?', p[2])
        gains.append(float(m.group(1)))
        bases.append(float(m.group(2)) if m.group(2) is not None else float(p[4]))
    raw = np.fromfile(os.path.join(os.path.dirname(path), dat), dtype='<i2').reshape(n_samp, n_sig)
    return ((raw - np.array(bases)) / np.array(gains)).T

def read_record(path):
    """[12, 1000] in physical units."""
    if wfdb is not None:
        return wfdb.rdsamp(path, channels=list(range(N_LEADS)))[0].T
    return _read_wfdb_numpy(path)

split = json.load(open(SPLIT_PATH))
meta = json.load(open(SPLIT_PATH.replace('.json', '.meta.json')))
sha = hashlib.sha256(open(SPLIT_PATH, 'rb').read()).hexdigest()
assert sha == meta['sha256'], 'split file differs from .meta.json: stop (README section 1)'

sys.path.insert(0, DATASET_DIR)
from make_split import load_single_label_records         # same label filter as the split
df = load_single_label_records(PTBXL_ROOT)
mk = np.load(MASK_PATH)
pool_ids = mk['ids']                                      # every single-label record, sorted ecg_id
assert len(pool_ids) == len(df) and (pool_ids == df.index.values).all(), 'PTB-XL copy differs from the one used for the split'
pool_y = df.label.values.astype(np.int64)

cache = f'{LOCAL_CACHE}/ptbxl_pool_{SEQ_LEN}.npy'
if os.path.exists(cache):
    X_np = np.load(cache)
    print('loaded cached signals', X_np.shape)
else:
    files = [os.path.join(PTBXL_ROOT, f) for f in df.filename_lr.values]
    X_np = np.empty((len(files), N_LEADS, SEQ_LEN), dtype=np.float32)
    t_old, t_new = np.linspace(0, 1, RAW_LEN), np.linspace(0, 1, SEQ_LEN)
    with ThreadPoolExecutor(16) as ex:
        for s in range(0, len(files), 1000):              # chunks keep RAM low
            raw = np.stack(list(ex.map(read_record, files[s:s + 1000]))).astype(np.float64)   # [n, 12, 1000]
            assert raw.shape[1:] == (N_LEADS, RAW_LEN), raw.shape
            c = interpolate.interp1d(t_old, raw, axis=-1)(t_new)                              # [n, 12, 608]
            X_np[s:s + len(c)] = (c - c.mean(-1, keepdims=True)) / (c.std(-1, keepdims=True) + 1e-8)
            print('read', s + len(c), '/', len(files))
    np.save(cache, X_np)
assert np.isfinite(X_np).all()

pos = {int(e): i for i, e in enumerate(pool_ids)}
idx_m, idx_v, idx_n = (np.array([pos[e] for e in split[k]]) for k in ('train', 'val', 'test'))
assert len(idx_m) == meta['sizes']['train'] and len(idx_n) == meta['sizes']['test']
masks_pool = mk['masks'][:K]                                        # (K, N) True = shadow j trains on it
assert (masks_pool.sum(0) == K // 2).all(), 'every record needs K/2 IN and K/2 OUT shadows: keep K = 8'
ref_pool = np.flatnonzero(np.isin(pool_ids, mk['ref_ids']))       # RMIA reference population Z

dev = os.environ.get('PTST_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
X = torch.as_tensor(X_np).to(dev)                                   # whole pool on the GPU (~0.5 GB): fast batching
Y = torch.as_tensor(pool_y).to(dev)
print('pool', tuple(X.shape), '| member', len(idx_m), 'val', len(idx_v), 'non-member', len(idx_n), '| device', dev,
      torch.cuda.get_device_name(0) if dev == 'cuda' else '')

## 4. Model PatchTST
PatchTST (Nie et al., 2023) cắt mỗi lead thành các **patch** dài 16 điểm, bước 8 (512 điểm → 63 patch), coi mỗi patch như một "từ" rồi cho qua Transformer encoder. Mỗi lead được xử lý độc lập với cùng trọng số (channel-independence). Bên trong có RevIN (`scaling='std'`): chuẩn hoá từng mẫu trước khi vào model và hoàn nguyên ở output fcst.

Kiến trúc cố định trước, **không tune** (dùng bản của HuggingFace `transformers`, train từ đầu): patch 16, stride 8, d_model 128, 16 head, 3 layer, FFN 256, dropout 0,2.
- cls: `PatchTSTForClassification` → 5 logit.
- fcst: `PatchTSTForPrediction` → 96 điểm × 12 lead.

Cell này cũng chạy thử 1 batch để kiểm tra shape trước khi train thật.

In [ ]:
from transformers import PatchTSTConfig, PatchTSTForClassification, PatchTSTForPrediction
ARCH = dict(num_input_channels=N_LEADS, context_length=LOOKBACK, patch_length=16, patch_stride=8,
            d_model=128, num_attention_heads=16, num_hidden_layers=3, ffn_dim=256,
            dropout=0.2, head_dropout=0.2, scaling='std')
BACKBONE = 'PatchTST (HF transformers, trained from scratch)'

def build_model():
    if TASK == 'cls':
        return PatchTSTForClassification(PatchTSTConfig(**ARCH, num_targets=len(CLASSES)))
    return PatchTSTForPrediction(PatchTSTConfig(**ARCH, prediction_length=HORIZON))

USE_BF16 = dev == 'cuda' and torch.cuda.get_device_capability()[0] >= 8   # T4 -> fp32; A100 / L4 -> bf16
def amp():
    if dev == 'cuda':
        return torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=USE_BF16)
    return torch.autocast(device_type='cpu', enabled=False)

def forward(model, xb):
    """xb: [B, 12, 608] -> cls logits [B, 5] | fcst predictions [B, 12, 96]. Only the first 512 points go in."""
    inp = xb[..., :LOOKBACK].transpose(1, 2).contiguous()           # PatchTST wants [B, L, C]
    with amp():
        if TASK == 'cls':
            return model(past_values=inp).prediction_logits.float()
        return model(past_values=inp).prediction_outputs.float().transpose(1, 2)

_m = build_model().to(dev).eval()
with torch.no_grad():
    print(TASK, '| params', sum(p.numel() for p in _m.parameters()), '| precision', 'bf16' if USE_BF16 else 'fp32',
          '| input', tuple(X[:4].shape), '-> output', tuple(forward(_m, X[:4]).shape))
del _m

## 5. Recipe huấn luyện chung
Một hàm `train()` dùng cho **cả tune, target và shadow** (điều kiện để LiRA / RMIA có nghĩa: shadow phải được train giống hệt target, chỉ khác tập dữ liệu và seed).
- batch 16, bỏ batch lẻ cuối, AdamW (lr đầu = `max_lr / 25`), OneCycleLR, grad clip 1.0, không early stopping, lấy model ở epoch cuối.
- `predict()` chạy model trên một tập chỉ số (mặc định cả pool) → logit (cls) hoặc dự báo (fcst).
- `utility()` đo chất lượng model: cls = accuracy, fcst = MSE. `gap` = chênh lệch member vs non-member: gap lớn = overfit = thường dễ bị MIA hơn.

In [ ]:
def control_randomness(seed):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def save_atomic(obj, path):          # write then rename: a cut session never leaves a half-written file
    torch.save(obj, path + '.tmp'); os.replace(path + '.tmp', path)

def batch_loss(out, rows):
    if TASK == 'cls':
        return F.cross_entropy(out, Y[rows])
    return F.mse_loss(out, X[rows][..., LOOKBACK:])

def train(rows, epochs, max_lr, seed, eval_at=(), on_eval=None, verbose=True):
    """Fresh model trained on pool rows `rows`. on_eval(model, ep, train_loss) runs after each epoch in eval_at."""
    control_randomness(seed)
    model = build_model().to(dev)
    rows = torch.as_tensor(np.asarray(rows), device=dev)
    steps = len(rows) // BATCH                                      # drop_last
    opt = torch.optim.AdamW(model.parameters(), lr=max_lr / 25)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, total_steps=epochs * steps)
    gen = torch.Generator().manual_seed(seed)
    for ep in range(1, epochs + 1):
        model.train(); t0, losses = time.time(), []
        perm = rows[torch.randperm(len(rows), generator=gen).to(dev)]
        for s in range(steps):
            b = perm[s * BATCH:(s + 1) * BATCH]
            opt.zero_grad(set_to_none=True)
            loss = batch_loss(forward(model, X[b]), b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step(); sched.step(); losses.append(loss.item())
        if verbose:
            print(f'  ep {ep}/{epochs} train_loss {np.mean(losses):.4f} ({time.time() - t0:.0f}s)')
        if ep in eval_at and on_eval:
            on_eval(model, ep, float(np.mean(losses)))
    return model

@torch.no_grad()
def predict(model, rows=None, bs=256):
    """cls -> logits [n, 5]; fcst -> predictions [n, 12, H]. rows default to the whole pool."""
    model.eval()
    rows = torch.arange(len(X), device=dev) if rows is None else torch.as_tensor(rows, device=dev)
    return torch.cat([forward(model, X[rows[i:i + bs]]).cpu() for i in range(0, len(rows), bs)])

def fcst_mse(preds, rows=None):
    """Per-record MSE of fcst predictions against the true last 96 points."""
    tgt = X[..., LOOKBACK:].cpu() if rows is None else X[torch.as_tensor(rows, device=dev)][..., LOOKBACK:].cpu()
    return ((preds.double() - tgt.double()) ** 2).mean(dim=(1, 2)).numpy()

def utility(out):
    """out over the whole pool: cls logits [N, 5] or fcst per-record MSE [N]."""
    if TASK == 'cls':
        hit = out.argmax(1).numpy() == pool_y
        a_m, a_n, a_v = (float(hit[i].mean()) for i in (idx_m, idx_n, idx_v))
        return {'train_acc': a_m, 'test_acc': a_n, 'val_acc': a_v, 'gap': a_m - a_n}
    m_m, m_n, m_v = (float(out[i].mean()) for i in (idx_m, idx_n, idx_v))
    return {'train_mse': m_m, 'test_mse': m_n, 'val_mse': m_v, 'gap': m_n - m_m}

def recipe_key(epochs, max_lr, extra=''):
    """Fingerprint of a run: a checkpoint is reused only if its key matches (same as v1, so v1 files work)."""
    return f"patchtst|{TASK}|{json.dumps(ARCH, sort_keys=True)}|{LEVEL}|{SEED}|{epochs}|{BATCH}|{max_lr}|{meta['sha256'][:12]}{extra}"

## 6. Tune trên `val` (≤ 6 cấu hình)
Thử từng cặp `(epochs, max_lr)`: train trên member, đo utility trên `val`, chọn cấu hình tốt nhất (cls: `val_acc` cao nhất, fcst: `val_mse` thấp nhất). **Không nhìn điểm MIA** khi chọn, nếu không kết quả tấn công sẽ bị "chọn lọc" (README mục 3).
Kết quả lưu ở `tuning.json`; chạy lại thì cấu hình đã xong được đọc lại.

In [ ]:
def tune():
    path = f'{CKPT_DIR}/tuning.json'
    done = json.load(open(path)) if os.path.exists(path) else {'configs': []}
    assert len(TUNE_GRID) <= 6, 'dataset/README.md allows at most 6 configs per model and task'
    for ep, lr in TUNE_GRID:
        if any(c['epochs'] == ep and c['max_lr'] == lr for c in done['configs']):
            continue
        print(f'[tune] epochs={ep} max_lr={lr}'); t0 = time.time()
        model = train(idx_m, ep, lr, SEED, verbose=False)
        rows = np.r_[idx_m, idx_v]; nm = len(idx_m)
        out = predict(model, rows)
        if TASK == 'cls':
            hit = out.argmax(1).numpy() == pool_y[rows]
            res = {'train_acc': float(hit[:nm].mean()), 'val_acc': float(hit[nm:].mean())}
        else:
            mse = fcst_mse(out, rows)
            res = {'train_mse': float(mse[:nm].mean()), 'val_mse': float(mse[nm:].mean())}
        done['configs'].append({'epochs': ep, 'max_lr': lr, **res})
        json.dump(done, open(path, 'w'), indent=1)
        print('  ', done['configs'][-1], f'{(time.time() - t0) / 60:.1f} min')
    metric = 'val_acc' if TASK == 'cls' else 'val_mse'
    in_grid = [c for c in done['configs'] if (c['epochs'], c['max_lr']) in TUNE_GRID]
    best = (max if TASK == 'cls' else min)(in_grid, key=lambda c: c[metric])
    done['chosen'] = {'epochs': best['epochs'], 'max_lr': best['max_lr'], 'by': f'best {metric} (val only, no MIA scores)'}
    json.dump(done, open(path, 'w'), indent=1)
    return done

if RUN_TUNE:
    tuning = tune()
    EPOCHS, MAX_LR = tuning['chosen']['epochs'], tuning['chosen']['max_lr']
    TUNING = tuning['configs']                                     # written into every JSON (extra.tuning)
    print(pd.DataFrame(TUNING).round(4).to_string(index=False))
else:
    TUNING = 'manual recipe, no search'
EVAL_AT = eval_epochs(EPOCHS)
KEY = recipe_key(EPOCHS, MAX_LR)
print('-> EPOCHS', EPOCHS, 'MAX_LR', MAX_LR, 'eval_at', EVAL_AT)

## 7. Target model
Train trên **member** với recipe đã chọn. Ở các epoch 10 / 20 / 50 / 100% (`EVAL_AT`), chạy model trên toàn pool và lưu output vào `target_ep<ep>.pt`:
- epoch cuối dùng cho mọi attack;
- các epoch giữa chỉ dùng cho Temporal cls (quỹ đạo loss qua quá trình train).

`target.done.json` ghi lại utility và gap ở từng epoch. Đã train rồi (key khớp) thì bỏ qua.

In [ ]:
def target_out_path(ep):
    return f'{CKPT_DIR}/target_ep{ep}.pt'

def run_target():
    done_path = f'{CKPT_DIR}/target.done.json'
    if os.path.exists(done_path) and json.load(open(done_path))['key'] == KEY:
        print('target already trained'); return json.load(open(done_path))['history']
    history = []
    def on_eval(model, ep, train_loss):
        out = predict(model)                                     # whole pool, incl. val
        rec = {'key': KEY, 'epoch': ep}
        if TASK == 'cls':
            rec['logits'] = out; u = utility(out)
        else:
            rec['preds'] = out.half()                            # [N, 12, H], half precision to save disk
            rec['mse'] = fcst_mse(out); u = utility(rec['mse'])
        save_atomic(rec, target_out_path(ep))
        history.append({'epoch': ep, 'train_loss': train_loss, **u})
        print('  eval', {k: round(v, 4) for k, v in history[-1].items()})
    t0 = time.time()
    train(idx_m, EPOCHS, MAX_LR, SEED, eval_at=EVAL_AT, on_eval=on_eval)
    json.dump({'key': KEY, 'recipe': {'epochs': EPOCHS, 'max_lr': MAX_LR}, 'eval_epochs': EVAL_AT, 'history': history},
              open(done_path, 'w'), indent=1)
    print(f'target done in {(time.time() - t0) / 60:.1f} min')
    return history

history = run_target()
target = torch.load(target_out_path(EPOCHS), weights_only=False)
UTIL = utility(target['logits'] if TASK == 'cls' else target['mse'])
print(pd.DataFrame(history).round(4).to_string(index=False))

## 8. 8 shadow model
Shadow `j` dùng **đúng recipe của target**, train trên các bản ghi có `masks_pool[j] = True` (một nửa pool), seed `42 + 1 + j`. Sau khi train, chạy trên toàn pool và chỉ lưu thống kê cần cho attack (cls: logit, fcst: MSE từng bản ghi) vào `shadow_k<j>.pt`.

Ý tưởng: với mỗi bản ghi, có 4 shadow **đã thấy** nó (IN) và 4 shadow **chưa thấy** (OUT). So output của target với hai nhóm này cho biết target "giống IN" hay "giống OUT". Shadow đã xong được bỏ qua.

In [ ]:
def run_shadow(j):
    path = f'{CKPT_DIR}/shadow_k{j}.pt'
    key = recipe_key(EPOCHS, MAX_LR, f"|shadow{j}|{meta['shadow']['sha256'][:12]}")
    if os.path.exists(path) and torch.load(path, weights_only=False)['key'] == key:
        print(f'shadow {j} already trained'); return
    t0, m = time.time(), masks_pool[j]
    print(f'[shadow {j}/{K}] training on {m.sum()} records')
    model = train(np.flatnonzero(m), EPOCHS, MAX_LR, SEED + 1 + j, verbose=False)
    out = predict(model)
    rec = {'key': key}
    if TASK == 'cls':
        rec['logits'] = out
        hit = out.argmax(1).numpy() == pool_y
        print(f'  acc IN {hit[m].mean():.4f}  OUT {hit[~m].mean():.4f}')
    else:
        rec['mse'] = fcst_mse(out)
        print(f'  mse IN {rec["mse"][m].mean():.4f}  OUT {rec["mse"][~m].mean():.4f}')
    save_atomic(rec, path)
    print(f'  done in {(time.time() - t0) / 60:.1f} min')

for j in range(K):
    run_shadow(j)

## 9. Metric và ghi JSON
Mỗi attack cho ra một score cho từng member và non-member. Từ đó:
- **TPR @ 0,1% FPR** và **TPR @ 1% FPR** (metric chính): chỉ cho phép báo nhầm 0,1% (hoặc 1%) non-member là member, thì bắt được bao nhiêu % member. Đây là câu hỏi "attack có *chắc chắn* lộ được ai không".
- **AUC**: chỉ để đối chiếu (AUC 0,5 = đoán bừa).
- **Bootstrap CI 95%** (1000 lần lấy mẫu lại) cho hai TPR, vì ở FPR 0,1% chỉ có ~8 non-member quyết định ngưỡng.

Schema theo `results/_sample.json`; thông tin recipe, các khác biệt so với README (`deviations`) và cam kết không chọn theo điểm MIA (`selection_note`) nằm trong `extra`.

In [ ]:
from sklearn.metrics import roc_curve, auc as sk_auc

def tpr_at(fpr, tpr, t):
    m = fpr <= t
    return float(tpr[m].max()) if m.any() else 0.0

def mia_metrics(sm, sn):
    fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm)), np.zeros(len(sn))], np.r_[sm, sn])
    return tpr_at(fpr, tpr, 1e-3), tpr_at(fpr, tpr, 1e-2), float(sk_auc(fpr, tpr))

def bootstrap_ci(sm, sn, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    res = np.array([mia_metrics(sm[rng.integers(0, len(sm), len(sm))], sn[rng.integers(0, len(sn), len(sn))])[:2] for _ in range(n)])
    lo, hi = np.percentile(res, [2.5, 97.5], axis=0)
    return [float(lo[0]), float(hi[0])], [float(lo[1]), float(hi[1])]

DEVIATIONS = [   # every difference from dataset/README.md sections 2 and 3 (same list as v1)
    'Mục 2: z-score chia cho (std + 1e-8) thay vì std, để tránh chia 0 ở lead phẳng; ảnh hưởng không đáng kể.',
    "Mục 2: PatchTST có RevIN bên trong model (scaling='std'): chuẩn hóa input từng mẫu và hoàn nguyên output fcst. "
    'Đây là một phần kiến trúc (Nie et al. 2023), áp dụng SAU tiền xử lý chung, không thay tiền xử lý chung.',
    'Mục 2: đọc PTB-XL bằng loader riêng (wfdb, dự phòng bằng trình đọc WFDB format 16 viết bằng numpy), không dùng momentfm PTBXL_dataset; '
    'cùng bộ lọc single-superclass của make_split.py và cùng nội suy tuyến tính 1000 -> 608. Thứ tự bản ghi theo ecg_id tăng dần.',
    'Mục 3: kiến trúc cố định trước, không tune: patch 16, stride 8, d_model 128, 16 head, 3 layer, FFN 256, dropout 0.2, head_dropout 0.2.',
    'Mục 3: AdamW dùng weight_decay mặc định của PyTorch (0.01), giống notebook MOMENT; không thêm regularization nào khác.',
    'Mục 3: precision fp32 vì GPU Kaggle T4 không hỗ trợ bf16 (đúng quy tắc "fp32 nếu GPU cũ"); A100/L4 sẽ tự dùng bf16.',
    'Mục 3: lưới tune 6 cấu hình epochs {20, 50} x max_lr {1e-4, 5e-4, 1e-3}; mỗi cấu hình train trên toàn bộ member với seed 42, '
    'đánh giá ở epoch cuối trên val; chọn cấu hình val tốt nhất (cls: accuracy cao nhất, fcst: MSE thấp nhất).',
]
SELECTION_NOTE = ('Không có bước chọn hyperparameter hay checkpoint nào nhìn vào điểm MIA hoặc non-member của test: '
                  'tune chỉ dùng utility trên train/val (xem extra.tuning), điểm MIA chỉ được tính sau khi target và shadow đã train xong, '
                  'shadow dùng đúng recipe cuối của target, tham số attack (tau, gamma, a, HistGradientBoosting) cố định trước.')
EXTRA = {'epochs': EPOCHS, 'eval_epoch': EPOCHS, 'eval_epochs': EVAL_AT, 'batch': BATCH, 'max_lr': MAX_LR,
         'backbone': BACKBONE, 'arch': ARCH,
         'optimizer': 'AdamW, OneCycleLR (lr0 = max_lr/25), grad clip 1.0, no early stopping',
         'tuning': TUNING, 'split_sha256': meta['sha256'],
         'preprocess': f'interp 1000->{SEQ_LEN}, z-score per lead per record, input = first {LOOKBACK}',
         'record_order': 'sorted ecg_id', 'deviations': DEVIATIONS, 'selection_note': SELECTION_NOTE,
         **({'lookback': LOOKBACK, 'horizon': HORIZON} if TASK == 'fcst' else {})}

def save_result(attack, sm, sn, extra=None, n_shadow=None):
    t01, t1, au = mia_metrics(sm, sn); c01, c1 = bootstrap_ci(sm, sn)
    out = {'model': 'patchtst', 'task': TASK, 'attack': attack, 'owner_tv': 2, 'level': LEVEL,
           'dataset': 'ptbxl', 'seed': SEED, 'n_member': int(len(sm)), 'n_nonmember': int(len(sn)),
           'scores_member': np.round(sm, 6).tolist(), 'scores_nonmember': np.round(sn, 6).tolist(),
           'tpr_at_0.1fpr': t01, 'tpr_at_1fpr': t1, 'auc': au,
           'bootstrap_ci': {'n_resamples': N_BOOT, 'confidence': 0.95, 'tpr_at_0.1fpr': c01, 'tpr_at_1fpr': c1},
           'utility': {k: v for k, v in UTIL.items() if not k.startswith('val_')},
           'n_shadow': n_shadow, 'extra': {**EXTRA, **(extra or {})}}
    json.dump(out, open(f'{RESULTS_DIR}/{attack}.json', 'w'))
    print(f'{attack:13s} TPR@0.1% {t01:.4f} {np.round(c01, 4).tolist()}  TPR@1% {t1:.4f} {np.round(c1, 4).tolist()}  AUC {au:.4f}')
    return out

## 10. Loss / Confidence / Entropy (chỉ cần target)
Attack đơn giản nhất: model thường "tự tin" hơn trên dữ liệu đã học.
- `loss` = −CE (cls) hoặc −MSE (fcst): loss thấp → giống member.
- `conf` = xác suất lớn nhất của softmax (chỉ cls).
- `entropy` = −H(p) (chỉ cls): phân phối càng nhọn → giống member.

Không cần shadow, nhưng một ngưỡng chung cho mọi bản ghi nên thường yếu ở FPR thấp: bản ghi "dễ" thì loss thấp dù là non-member.

In [ ]:
im, inn = idx_m, idx_n
if TASK == 'cls':
    logp = F.log_softmax(target['logits'].double(), -1); p = logp.exp()
    ce = -logp.gather(1, torch.as_tensor(pool_y)[:, None]).squeeze(1)
    basic = {'loss': (-ce).numpy(), 'conf': p.max(1).values.numpy(), 'entropy': (p * logp).sum(1).numpy()}
else:
    basic = {'loss': -target['mse']}
for a, s in basic.items():
    r = save_result(a, s[im], s[inn])
    if a == 'loss' and r['auc'] <= 0.5:
        print('WARNING: Loss attack AUC <= 0.5; check the score direction and whether the model learned anything')
print('utility', {k: round(v, 4) for k, v in UTIL.items()})

## 11. LiRA và RMIA (cần 8 shadow)
Cả hai đều so target với shadow **trên từng bản ghi**, nên sửa được điểm yếu "bản ghi dễ" của bước 10.

Thống kê dùng chung cho mỗi bản ghi:
- cls: `φ = log(p / (1 − p))` của nhãn đúng (logit-scaled confidence), `p` = softmax của nhãn đúng.
- fcst: `φ = −log(MSE)`, `p = exp(−MSE / τ)` với τ = trung vị MSE của shadow OUT (chỉ dùng thông tin phía attacker).

**LiRA** (Carlini et al. 2022): ước lượng phân phối Gauss của φ ở 4 shadow IN và 4 shadow OUT (phương sai toàn cục vì chỉ có 4 mẫu mỗi bên).
- `lira` (online, số chính): log-likelihood ratio `log N(φ_target; IN) − log N(φ_target; OUT)`.
- `lira_offline`: chỉ dùng OUT, z-score `(φ_target − μ_OUT) / σ_OUT`.

**RMIA** (Zarifzadeh et al. 2024): tỉ số `p_target(x) / p̄(x)` rồi so với một quần thể tham chiếu Z (5000 bản ghi): score = tỉ lệ z ∈ Z mà tỉ số của x lớn hơn γ lần tỉ số của z (γ = 2).
- `rmia` (offline, số chính): `p̄ = ½[(1 + a)·p_OUT + (1 − a)]`, a = 0,3.
- `rmia_online`: `p̄ = ½(p_IN + p_OUT)`.

In [ ]:
from scipy.stats import norm
shadows = [torch.load(f'{CKPT_DIR}/shadow_k{j}.pt', weights_only=False) for j in range(K)]
Min = masks_pool.astype(float)                                      # 1 = shadow j saw record i
idx_t = np.r_[idx_m, idx_n]; n_m = len(idx_m)                       # attacked records: members first
RMIA_A, RMIA_GAMMA = meta['shadow']['rmia_defaults']['a'], meta['shadow']['rmia_defaults']['gamma']

def phi_p(logits, y):
    """phi = log(p/(1-p)) of the true class (numerically stable), p = softmax prob of the true class."""
    z = logits.double(); y = torch.as_tensor(y).long()[:, None]
    phi = z.gather(1, y).squeeze(1) - torch.logsumexp(z.scatter(1, y, float('-inf')), 1)
    return phi.numpy(), torch.sigmoid(phi).numpy()

def group_stats(W, V):               # per-record mean / unbiased variance over the shadows selected by W
    c = W.sum(0); mu = (W * V).sum(0) / c
    return mu, (W * (V - mu) ** 2).sum(0) / (c - 1)

def lira_scores(PHI_S, PHI_T, online):
    mu_in, v_in = group_stats(Min, PHI_S); mu_out, v_out = group_stats(1 - Min, PHI_S)
    s_in, s_out = np.sqrt(v_in[idx_t].mean()), np.sqrt(v_out[idx_t].mean())     # global std (few shadows per sample)
    x = PHI_T[idx_t]
    if online:
        return norm.logpdf(x, mu_in[idx_t], s_in) - norm.logpdf(x, mu_out[idx_t], s_out)
    return (x - mu_out[idx_t]) / s_out

def rmia_scores(P_S, P_T, online, a=RMIA_A, gamma=RMIA_GAMMA):
    Mout = 1 - Min
    p_in, p_out = (Min * P_S).sum(0) / Min.sum(0), (Mout * P_S).sum(0) / Mout.sum(0)
    p_bar = 0.5 * (p_in + p_out) if online else 0.5 * ((1 + a) * p_out + (1 - a))
    ratio = P_T / p_bar
    z = ratio[np.intersect1d(ref_pool, idx_t)]                     # Z: reference records the target was scored on
    return np.searchsorted(np.sort(z), ratio[idx_t] / gamma, side='right') / len(z)

rmia_note, att_extra = '', {}
if TASK == 'cls':
    PHI_S, P_S = map(np.stack, zip(*[phi_p(s['logits'], pool_y) for s in shadows]))   # [K, N]
    PHI_T, P_T = phi_p(target['logits'], pool_y)                                       # [N]
else:
    S_mse = np.stack([s['mse'] for s in shadows]).astype(np.float64)
    T_mse = target['mse'].astype(np.float64)
    PHI_S, PHI_T = -np.log(S_mse + 1e-8), -np.log(T_mse + 1e-8)
    TAU = float(np.median(S_mse[~masks_pool]))                     # attacker-side only (shadow OUT)
    P_S, P_T = np.exp(-S_mse / TAU), np.exp(-T_mse / TAU)
    rmia_note = f' p=exp(-MSE/tau) tau={TAU:.5f}'
    sens = {}                                                       # diagnostic only: the files use tau x1.0
    for tm in (0.5, 1.0, 2.0):
        for online in (False, True):
            s = rmia_scores(np.exp(-S_mse / (TAU * tm)), np.exp(-T_mse / (TAU * tm)), online)
            sens[f"{'online' if online else 'offline'}_x{tm}"] = mia_metrics(s[:n_m], s[n_m:])
    att_extra = {'tau': TAU, 'rmia_tau_sensitivity': sens}

scores = {'lira': lira_scores(PHI_S, PHI_T, True), 'lira_offline': lira_scores(PHI_S, PHI_T, False),
          'rmia': rmia_scores(P_S, P_T, False), 'rmia_online': rmia_scores(P_S, P_T, True)}
method = {'lira': 'LiRA online, global variance', 'lira_offline': 'LiRA offline z-score, global variance',
          'rmia': f'RMIA offline a={RMIA_A} gamma={RMIA_GAMMA}{rmia_note}', 'rmia_online': f'RMIA online gamma={RMIA_GAMMA}{rmia_note}'}
r_off = scores['rmia']
print(f'RMIA offline: {np.mean(r_off == 0):.1%} scores = 0, {np.mean(r_off == 1):.1%} = 1 (near 100% at 0 means gamma saturates)')
for a, s in scores.items():
    save_result(a, s[:n_m], s[n_m:], {**att_extra, 'method': method[a], 'shadow_masks_sha256': meta['shadow']['sha256']}, n_shadow=K)

## 12. Temporal (gray-box)
Thay vì một con số, lấy **cả một vector đặc trưng** cho mỗi bản ghi rồi train một classifier member / non-member (HistGradientBoosting). Mạnh hơn nhưng giả định attacker biết thêm (gray-box): có một ít bản ghi đã biết nhãn member / non-member, và với cls là thấy được các checkpoint giữa quá trình train.
- **cls** (training dynamics): CE / entropy / margin ở 4 epoch `EVAL_AT`, mức giảm CE giữa các epoch, độ dốc CE và margin.
- **fcst** (Koren et al.): phần dư `residual = thật − dự báo`: biên độ DFT ở 8 dải tần, trend, drift, log-MSE theo từng lead.

**Cross-fitting 5-fold**: classifier học trên 4 phần, chấm điểm phần còn lại, nên không bản ghi nào được chấm bởi classifier đã thấy nó. Bảng ablation in ra so sánh với các biến thể đơn giản hơn (logistic regression, chỉ epoch cuối, chỉ loss) để xem đặc trưng nào thực sự giúp; chỉ `temporal` được ghi file.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def cross_fit(make_clf, Xf, y):
    score = np.zeros(len(y))
    for tr, te in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(Xf, y):
        score[te] = make_clf().fit(Xf[tr], y[tr]).predict_proba(Xf[te])[:, 1]
    return score
hgb = lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=SEED)
logreg = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
def slope_last(v):                   # least-squares slope along the last axis
    t = np.arange(v.shape[-1], dtype=np.float64); t -= t.mean()
    return (v * t).sum(-1) / (t ** 2).sum()

y_t = np.r_[np.ones(n_m), np.zeros(len(idx_n))]
if TASK == 'cls':
    def per_epoch_cls(logits, labels):
        logp = F.log_softmax(logits.double(), -1); lab = torch.as_tensor(labels)[:, None]
        ce = -logp.gather(1, lab).squeeze(1); ent = -(logp.exp() * logp).sum(1)
        true = logits.double().gather(1, lab).squeeze(1)
        other = logits.double().scatter(1, lab, float('-inf')).max(1).values
        return ce.numpy(), ent.numpy(), (true - other).numpy()
    recs = {ep: torch.load(target_out_path(ep), weights_only=False)['logits'] for ep in EVAL_AT}
    ce, ent, mar = (np.stack(z, 1) for z in zip(*[per_epoch_cls(recs[ep][idx_t], pool_y[idx_t]) for ep in EVAL_AT]))  # [n, 4]
    t = np.asarray(EVAL_AT, dtype=np.float64); t -= t.mean()
    slope_ep = lambda v: (v * t).sum(1) / (t ** 2).sum()
    Xf = np.concatenate([ce, ent, mar, np.diff(ce, axis=1), np.stack([slope_ep(ce), slope_ep(mar)], 1)], 1)
    loss_all = -ce[:, -1]
    makers = {'temporal': (hgb, Xf), 'temporal_lr': (logreg, Xf),
              'final_epoch_clf': (hgb, np.stack([ce[:, -1], ent[:, -1], mar[:, -1]], 1)), 'loss_only_clf': (hgb, loss_all[:, None])}
    t_extra = {'variant': 'training-dynamics', 'eval_epochs': EVAL_AT,
               'features': 'per-epoch CE/entropy/margin + consecutive CE change + CE and margin slope over checkpoints',
               'threat_model': 'gray-box: intermediate checkpoints, true labels and a labelled member/non-member subset'}
else:
    res = X_np[idx_t][..., LOOKBACK:].astype(np.float32) - target['preds'][idx_t].float().numpy()   # residual [n, 12, H]
    mag = np.abs(np.fft.rfft(res, axis=-1))[..., 1:]                # drop DC (= bias)
    dft = np.stack([np.log1p(mag[..., b].mean(-1)) for b in np.array_split(np.arange(mag.shape[-1]), N_BANDS)], -1).reshape(len(res), -1)
    Xf = np.concatenate([dft, slope_last(res), slope_last(np.abs(res)), np.log((res ** 2).mean(-1) + 1e-8)], 1)
    loss_all = -(res ** 2).mean((1, 2))
    makers = {'temporal': (hgb, Xf), 'temporal_lr': (logreg, Xf), 'loss_only_clf': (hgb, loss_all[:, None])}
    t_extra = {'variant': 'residual DFT (Koren et al.)', 'n_bands': N_BANDS,
               'features': f'DFT {N_BANDS} bands + trend + drift + per-lead log MSE of the residual',
               'threat_model': 'gray-box: a labelled member/non-member subset'}
print('features', Xf.shape)
S_t, M_t = {}, {}
for name, (mk_clf, feats) in makers.items():
    S_t[name] = cross_fit(mk_clf, feats, y_t)
    M_t[name] = mia_metrics(S_t[name][:n_m], S_t[name][n_m:])
M_t['loss (raw)'] = mia_metrics(loss_all[:n_m], loss_all[n_m:])
print('ablation\n' + pd.DataFrame(M_t, index=['TPR@0.1%', 'TPR@1%', 'AUC']).T.round(4).to_string())
save_result('temporal', S_t['temporal'][:n_m], S_t['temporal'][n_m:],
            {**att_extra, **t_extra, 'classifier': 'HistGradientBoosting(max_iter=200, lr=0.05)', 'n_folds': N_FOLDS,
             'ablation': {k: {'tpr_at_0.1fpr': v[0], 'tpr_at_1fpr': v[1], 'auc': v[2]} for k, v in M_t.items()}})

## 13. Tóm tắt + ROC log–log
ROC vẽ trên trục log để nhìn rõ vùng FPR thấp (0,1% – 1%), nơi metric chính nằm. Đường chéo nét đứt = đoán bừa; đường càng nằm trên cao ở bên trái càng nguy hiểm.

In [ ]:
import matplotlib.pyplot as plt
names = ['loss', 'conf', 'entropy', 'lira', 'lira_offline', 'rmia', 'rmia_online', 'temporal']
fig, ax = plt.subplots(figsize=(6, 6)); tab = []
for a in names:
    p = f'{RESULTS_DIR}/{a}.json'
    if not os.path.exists(p): continue
    r = json.load(open(p)); sm_, sn_ = np.array(r['scores_member']), np.array(r['scores_nonmember'])
    fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm_)), np.zeros(len(sn_))], np.r_[sm_, sn_])
    ax.plot(fpr, tpr, label=f"{a} (TPR@0.1% {r['tpr_at_0.1fpr']:.3f}, AUC {r['auc']:.3f})")
    tab.append({'attack': a, 'tpr@0.1%': r['tpr_at_0.1fpr'], 'CI@0.1%': np.round(r['bootstrap_ci']['tpr_at_0.1fpr'], 4).tolist(),
                'tpr@1%': r['tpr_at_1fpr'], 'CI@1%': np.round(r['bootstrap_ci']['tpr_at_1fpr'], 4).tolist(), 'auc': r['auc']})
ax.plot([1e-4, 1], [1e-4, 1], 'k--', lw=0.8)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(1e-4, 1); ax.set_ylim(1e-4, 1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=7); ax.set_title(f'PatchTST PTB-XL {TASK}, {LEVEL}-level')
plt.savefig(f'{RESULTS_DIR}/patchtst_{TASK}_roc.png', dpi=150, bbox_inches='tight'); plt.show()
print(pd.DataFrame(tab).round(4).to_string(index=False)); print('utility', {k: round(v, 4) for k, v in UTIL.items()})

## 14. Hình minh hoạ cho demo
Hai bản ghi: member "lộ" nhất (LiRA online cao nhất) và một non-member điển hình (LiRA gần 0). Bên trái: tín hiệu lead II. Bên phải: φ của 4 shadow IN (xanh), 4 shadow OUT (đỏ) và của target (nét đứt). Member bị lộ khi φ của target nằm hẳn về phía cụm IN.

In [ ]:
def show_record(i_pool, title, ax_sig, ax_phi):
    ax_sig.plot(X_np[i_pool, 1], lw=0.8); ax_sig.set_title(f'{title}: ecg_id {pool_ids[i_pool]}, lead II')
    if TASK == 'fcst': ax_sig.axvline(LOOKBACK, c='gray', ls=':', lw=0.8)   # right of the line = forecast target
    pin, pout = PHI_S[masks_pool[:, i_pool], i_pool], PHI_S[~masks_pool[:, i_pool], i_pool]
    ax_phi.scatter(pin, np.zeros_like(pin) + 1, label='shadow IN', c='tab:green')
    ax_phi.scatter(pout, np.zeros_like(pout), label='shadow OUT', c='tab:red')
    ax_phi.axvline(PHI_T[i_pool], c='k', ls='--', label='target model')
    ax_phi.set_yticks([0, 1]); ax_phi.set_yticklabels(['OUT', 'IN']); ax_phi.set_xlabel('phi'); ax_phi.legend(fontsize=7)
lira_on = scores['lira']
i_mem = idx_m[np.argmax(lira_on[:n_m])]                # most exposed member
i_non = idx_n[np.argmin(np.abs(lira_on[n_m:]))]        # typical non-member
fig, axs = plt.subplots(2, 2, figsize=(12, 5))
show_record(i_mem, 'MEMBER', axs[0, 0], axs[0, 1]); show_record(i_non, 'NON-MEMBER', axs[1, 0], axs[1, 1])
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/demo_lira_example.png', dpi=150); plt.show()

## 15. Đóng gói kết quả
Tải `patchtst_v2_results_<task>.zip` ở tab *Output*, chép vào **gốc repo MIA** rồi chọn **Extract Here** (không tạo thêm thư mục):
- `results/patchtst/<task>/*.json`, ROC, hình demo → kết quả MIA
- `experiment/patchtst_v2/run_artifacts/<task>/tuning.json`, `target.done.json` → bằng chứng tune và utility theo epoch

In [ ]:
STAGE = f'{WORK_ROOT}/_zip_{TASK}'
shutil.rmtree(STAGE, ignore_errors=True)
shutil.copytree(RESULTS_DIR, f'{STAGE}/results/patchtst/{TASK}')
art = f'{STAGE}/experiment/patchtst_v2/run_artifacts/{TASK}'
os.makedirs(art, exist_ok=True)
for f in ('tuning.json', 'target.done.json'):
    if os.path.exists(f'{CKPT_DIR}/{f}'):
        shutil.copy(f'{CKPT_DIR}/{f}', art)
z = shutil.make_archive(f'{WORK_ROOT}/patchtst_v2_results_{TASK}', 'zip', STAGE)
shutil.rmtree(STAGE)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')